# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsserGharib1/flyrank-internshipML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup, and the same slice as last week

Lane 2 stays locked. The feature window is March 2026, the later outcome is April 2026, and the decision date is 1 April. The lane guide lists CTR/engagement review context as a valid Lane 2 baseline idea, so this baseline intentionally tests one narrow, readable cue. The later action playbook can be broader.

In [2]:
%pip -q install duckdb pandas

import json, os, subprocess, sys
from pathlib import Path
import duckdb, pandas as pd, numpy as np

REPO_URL = "https://github.com/AsserGharib1/flyrank-internshipML"
REPO_DIR = "flyrank-internshipML"
IN_COLAB = "google.colab" in sys.modules

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "work").is_dir() and (candidate / "skills").is_dir():
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None and IN_COLAB:
    clone_root = Path("/content") / REPO_DIR
    if not clone_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_root)], check=True)
    repo_root = find_repo_root(clone_root)

assert repo_root is not None, "Could not find the FlyRank repository root."
os.chdir(repo_root)
OUTPUT_DIR = repo_root / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Add a Hugging Face read token as a Colab secret named HF_TOKEN."

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = BASE + "/fact_content_daily_performance"
FEATURE_MONTH, OUTCOME_MONTH = "2026-03", "2026-04"
DECISION_DATE = pd.Timestamp("2026-04-01")
FLOOR = 100

def month_rel(month):
    return f"read_parquet('{FACT}/month={month}/*.parquet')"

print("Repository root:", repo_root)
print("Writing outputs to:", OUTPUT_DIR)

Repository root: /content/flyrank-internshipML
Writing outputs to: /content/flyrank-internshipML/work/outputs


In [3]:
# Rebuild the same March -> April frame used in ML-04.
frame = con.sql(f"""
    WITH march AS (
        SELECT client_hash_id AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_prev30,
               SUM(gsc_clicks) AS clicks_prev30,
               COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_prev30,
               SUM(gsc_avg_position * gsc_impressions)
                 FILTER (WHERE gsc_avg_position > 0) * 1.0
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0), 0)
                 AS avg_position_prev30
        FROM {month_rel(FEATURE_MONTH)}
        GROUP BY 1, 2
    ),
    april AS (
        SELECT client_hash_id AS client_id,
               content_hash_id AS content_id,
               SUM(gsc_impressions) AS impressions_next30
        FROM {month_rel(OUTCOME_MONTH)}
        GROUP BY 1, 2
    )
    SELECT m.*, COALESCE(a.impressions_next30, 0) AS impressions_next30
    FROM march m
    LEFT JOIN april a USING (client_id, content_id)
    WHERE m.impressions_prev30 >= {FLOOR}
    ORDER BY client_id, content_id
""").df()

frame["change_pct"] = (
    (frame["impressions_next30"] - frame["impressions_prev30"])
    / frame["impressions_prev30"] * 100
)
frame["gap_vs_client"] = (
    frame["change_pct"] - frame.groupby("client_id")["change_pct"].transform("median")
)
frame["fell_behind"] = (frame["gap_vs_client"] <= -20).astype(int)
FRAME_BASE_RATE = frame["fell_behind"].mean()

print(f"Frame: {len(frame):,} pages, {frame.client_id.nunique()} clients")
print(f"Frame target base rate: {FRAME_BASE_RATE:.3f}")
print(f"Rows with average position below 1: {(frame.avg_position_prev30 < 1).sum():,}")

Frame: 101,441 pages, 44 clients
Frame target base rate: 0.290
Rows with average position below 1: 578


## 1. My rule and its reason code

I check two signals before writing the rule. Staleness is tied to FlyRank refresh logic. CTR relative to position is tied to CTR-fix logic. A negative result is allowed: if a field is not valid at the decision time, it stays out of the rule.

In [4]:
# SIGNAL 1 — can content_updated_date represent staleness on 1 April?
content = con.sql(f"""
    SELECT content_hash_id AS content_id, content_updated_date
    FROM read_parquet('{BASE}/dim_content.parquet')
""").df()

sig1 = frame.merge(content, on="content_id", how="left")
sig1["updated"] = pd.to_datetime(sig1["content_updated_date"])
sig1["days_since_update_at_decision"] = (DECISION_DATE - sig1["updated"]).dt.days

print(f"Rows dated after 1 April: {(sig1.updated > DECISION_DATE).mean():.1%}")
print(f"Median days since update at decision: {sig1.days_since_update_at_decision.median():.0f}")

Rows dated after 1 April: 82.1%
Median days since update at decision: -61


In [5]:
sig1["staleness_bucket"] = pd.cut(
    sig1["days_since_update_at_decision"],
    bins=[-10_000, -1, 30, 90, 180, 10_000],
    labels=["future-dated", "0-30 days", "31-90 days", "91-180 days", "180+ days"],
)

tbl1 = (
    sig1.groupby("staleness_bucket", observed=False)
        .agg(n=("fell_behind", "size"), fell_behind_rate=("fell_behind", "mean"))
        .round(3)
)
print(tbl1.to_string())
print(f"Frame base rate: {FRAME_BASE_RATE:.3f}")

                      n  fell_behind_rate
staleness_bucket                         
future-dated      83330             0.271
0-30 days            96             0.250
31-90 days        17873             0.378
91-180 days         124             0.419
180+ days            18             0.278
Frame base rate: 0.290


**Signal 1 verdict: FALSE.** The verdict is about using this field at the 1 April decision point, not about whether staleness matters in general. In the saved run, most values are dated after the decision date, so this field is not a valid feature for this baseline and stays out.

In [6]:
# SIGNAL 2 — CTR relative to pages in the same March position band.
missing_position = frame["avg_position_prev30"].isna()
below_one = frame["avg_position_prev30"] < 1
above_thirty = frame["avg_position_prev30"] > 30
print(f"Excluded rows without a positive position: {missing_position.sum():,}")
print(f"Excluded below-position-1 rows: {below_one.sum():,}")
print(f"Excluded rows below the top 30 positions: {above_thirty.sum():,}")

sig2 = frame[
    frame["avg_position_prev30"].notna()
    & frame["avg_position_prev30"].between(1, 30, inclusive="both")
].copy()
sig2["ctr_prev30"] = sig2["clicks_prev30"] / sig2["impressions_prev30"] * 100
sig2["position_band"] = pd.cut(
    sig2["avg_position_prev30"],
    bins=[1, 3, 5, 10, 20, 30],
    labels=["1-3", "3-5", "5-10", "10-20", "20-30"],
    include_lowest=True,
)

curve = (
    sig2.groupby("position_band", observed=False)
        .agg(n=("ctr_prev30", "size"), clicks=("clicks_prev30", "sum"), impressions=("impressions_prev30", "sum"))
)
curve["ctr_pct"] = (curve["clicks"] / curve["impressions"] * 100).round(2)
print("CTR by position band")
print(curve[["n", "ctr_pct"]].to_string())

band_median = sig2.groupby("position_band", observed=False)["ctr_prev30"].transform("median")
sig2["ctr_gap"] = sig2["ctr_prev30"] - band_median
sig2["ctr_gap_bucket"] = pd.qcut(
    sig2["ctr_gap"], 5,
    labels=["worst fifth", "second", "middle", "fourth", "best fifth"],
    duplicates="drop",
)
tbl2 = (
    sig2.groupby("ctr_gap_bucket", observed=False)
        .agg(n=("fell_behind", "size"), fell_behind_rate=("fell_behind", "mean"))
        .round(3)
)
print("\nLater outcome by CTR-gap bucket")
print(tbl2.to_string())

Excluded rows without a positive position: 0
Excluded below-position-1 rows: 578
Excluded rows below the top 30 positions: 13,345
CTR by position band
                   n  ctr_pct
position_band                
1-3             9177     0.40
3-5            17084     0.35
5-10           30726     0.30
10-20          19721     0.32
20-30          10810     0.19

Later outcome by CTR-gap bucket
                    n  fell_behind_rate
ctr_gap_bucket                         
worst fifth     17504             0.406
second          21124             0.327
middle          13884             0.272
fourth          17505             0.222
best fifth      17501             0.183


**Signal 2 verdict: CONFIRMED.** In the saved run, the worst CTR-gap fifth has a higher later `fell_behind` rate than the best fifth. The observed warehouse CTR values are low across all position bands, so I treat position as a comparison control rather than claim a large position effect. This is an association, not evidence that changing a title causes recovery.

**Rule:** among pages in positions 1–30, keep only pages below their position-band median CTR, then rank them by CTR shortfall weighted by March impressions.

**Reason code:** `below_ctr_for_position_band`

**Action label:** `review_title_and_meta`

## 2. Build the ranked queue

The rule uses March data only and has no fitted weights. Pages at or above their position-band median do not enter this action queue. The CSV is regenerated under the repository's `work/outputs/` directory.

In [7]:
q = sig2.copy()
q["ctr_shortfall"] = (-q["ctr_gap"]).clip(lower=0)
q["visibility"] = np.log1p(q["impressions_prev30"])
q["score"] = q["ctr_shortfall"] * q["visibility"]
q["reason_code"] = "below_ctr_for_position_band"
q["action_label"] = "review_title_and_meta"

# The reason code must be true for every row that enters the action queue.
q = q[q["ctr_shortfall"] > 0].copy()
queue = q.sort_values(
    ["score", "impressions_prev30", "client_id", "content_id"],
    ascending=[False, False, True, True],
    kind="mergesort",
).reset_index(drop=True)
queue["rank"] = queue.index + 1
QUEUE_BASE_RATE = queue["fell_behind"].mean()
assert queue["ctr_gap"].lt(0).all(), "Every queued row must be below its position-band median CTR."

COLS = [
    "rank", "score", "reason_code", "action_label", "client_id", "content_id",
    "impressions_prev30", "clicks_prev30", "ctr_prev30", "avg_position_prev30",
    "position_band", "ctr_gap", "active_days_prev30",
]
CSV_PATH = OUTPUT_DIR / "baseline_action_score.csv"
queue[COLS].to_csv(CSV_PATH, index=False, float_format="%.6f")

print(f"Queue: {len(queue):,} rows, {queue.client_id.nunique()} clients")
print(f"Queue base rate: {QUEUE_BASE_RATE:.3f}")
print("Wrote:", CSV_PATH)

Queue: 43,748 rows, 43 clients
Queue base rate: 0.354
Wrote: /content/flyrank-internshipML/work/outputs/baseline_action_score.csv


In [8]:
def precision_at_k(ranked, label, k):
    return float(ranked.head(k)[label].mean())

print(f"Queue base rate: {QUEUE_BASE_RATE:.3f}")
pk = {}
for k in (10, 20, 50, 100):
    p = precision_at_k(queue, "fell_behind", k)
    pk[f"p_at_{k}"] = round(p, 4)
    print(f"Precision@{k}: {p:.3f} | lift vs queue base: {p / QUEUE_BASE_RATE:.1f}x")

metrics = {
    "slice": {
        "feature_month": FEATURE_MONTH,
        "outcome_month": OUTCOME_MONTH,
        "decision_date": str(DECISION_DATE.date()),
        "min_march_impressions": FLOOR,
        "frame_rows": int(len(frame)),
        "frame_clients": int(frame.client_id.nunique()),
        "queue_rows": int(len(queue)),
        "queue_clients": int(queue.client_id.nunique()),
    },
    "label": {
        "definition": "April change at least 20 points below the median page on the same client",
        "queue_base_rate": round(float(QUEUE_BASE_RATE), 4),
    },
    "rule": {
        "eligibility": "position 1-30 and CTR below the March position-band median",
        "score": "ctr_shortfall_vs_position_band * log1p(march_impressions)",
        "reason_code": "below_ctr_for_position_band",
        "action_label": "review_title_and_meta",
    },
    "signal_verdicts": {"staleness": "FALSE", "ctr_vs_position": "CONFIRMED"},
    "precision_at_k": pk,
}

METRICS_PATH = OUTPUT_DIR / "baseline_metrics.json"
with METRICS_PATH.open("w") as handle:
    json.dump(metrics, handle, indent=2)
print("Wrote:", METRICS_PATH)
print("This is the full-slice baseline receipt. ML-08 must compare model and baseline on the same held-out split.")

Queue base rate: 0.354
Precision@10: 0.700 | lift vs queue base: 2.0x
Precision@20: 0.600 | lift vs queue base: 1.7x
Precision@50: 0.580 | lift vs queue base: 1.6x
Precision@100: 0.530 | lift vs queue base: 1.5x
Wrote: /content/flyrank-internshipML/work/outputs/baseline_metrics.json
This is the full-slice baseline receipt. ML-08 must compare model and baseline on the same held-out split.


## 3. Top-10 review

The live ML-07 card requires ten rows. Each row below includes the action, reason, evidence supporting its rank, and a condition that would make the recommendation wrong.

In [9]:
top10 = queue.head(10).copy()

def why_here(row):
    return (
        f"CTR {row.ctr_prev30:.3f}% is {abs(row.ctr_gap):.3f} points below its "
        f"position-band median with {row.impressions_prev30:,.0f} impressions"
    )

def confidence_note(row):
    return f"{row.impressions_prev30:,.0f} impressions across {int(row.active_days_prev30)} active days"

def wrong_if(row):
    if row.active_days_prev30 < 25:
        return "March activity is too intermittent for a stable peer comparison"
    if row.clicks_prev30 == 0:
        return "the query mix makes a click unnecessary or irrelevant"
    return "low CTR reflects query intent rather than a fixable snippet/content issue"

review = pd.DataFrame({
    "rank": top10["rank"],
    "action": top10["action_label"],
    "reason_code": top10["reason_code"],
    "why_here": [why_here(row) for _, row in top10.iterrows()],
    "confidence_note": [confidence_note(row) for _, row in top10.iterrows()],
    "wrong_if": [wrong_if(row) for _, row in top10.iterrows()],
})

print(review.to_string(index=False))
print(f"\nLater outcome check: {top10.fell_behind.sum()} of 10 met the target.")

 rank                action                 reason_code                                                                           why_here                           confidence_note                                                                  wrong_if
    1 review_title_and_meta below_ctr_for_position_band CTR 0.001% is 0.232 points below its position-band median with 134,984 impressions 134,984 impressions across 31 active days low CTR reflects query intent rather than a fixable snippet/content issue
    2 review_title_and_meta below_ctr_for_position_band  CTR 0.000% is 0.233 points below its position-band median with 44,707 impressions  44,707 impressions across 20 active days           March activity is too intermittent for a stable peer comparison
    3 review_title_and_meta below_ctr_for_position_band  CTR 0.000% is 0.232 points below its position-band median with 38,865 impressions  38,865 impressions across 31 active days                     the query mix makes a click unnece

The review is generated from the current top ten, so its evidence cannot drift away from the rows after a scoring change. The later outcome is shown only after ranking for evaluation. It is never part of the rule.

## 4. Weak picks + leakage check

The baseline has known limits. It cannot see query intent, it can concentrate recommendations within a few clients, and it excludes positions outside the interpretable 1–30 slice. These are reasons for human review, not reasons to hide the baseline.

In [10]:
top10_false_positives = int((top10["fell_behind"] == 0).sum())
print(f"Top-10 false positives under the later label: {top10_false_positives}")
print(f"Clients represented in the top 10: {top10.client_id.nunique()}")
print(f"Lowest position in the queue: {queue.avg_position_prev30.min():.2f}")

BUILT_FROM = ["clicks_prev30", "impressions_prev30", "avg_position_prev30"]
FORBIDDEN = ["impressions_next30", "change_pct", "gap_vs_client", "fell_behind", "content_updated_date"]
leaked = [column for column in FORBIDDEN if column in BUILT_FROM]
outcome_in_csv = [column for column in FORBIDDEN if column in COLS]

print("Rule inputs:", BUILT_FROM)
print("Forbidden future/label fields used by rule:", leaked)
print("Outcome fields written to reviewer CSV:", outcome_in_csv)

Top-10 false positives under the later label: 3
Clients represented in the top 10: 6
Lowest position in the queue: 1.00
Rule inputs: ['clicks_prev30', 'impressions_prev30', 'avg_position_prev30']
Forbidden future/label fields used by rule: []
Outcome fields written to reviewer CSV: []


## Self-check

- [x] Two signal checks have bucket counts and one-word verdicts
- [x] One transparent rule produces a score, reason code, and action label
- [x] Queue and metrics write under the repository `work/outputs/` path
- [x] Top 10 rows are reviewed from their current values
- [x] Rule inputs contain no future-window or label-derived fields
- [ ] Final corrected notebook committed and repo URL submitted on the ML-07 card